# Complete BambooAI Conversation Example

This notebook walks through a real BambooAI run, including environment setup, data loading, and the `pd_agent_converse` execution loop.

## Highlights

- Load the `testdata.csv` dataset that ships with this tutorial.
- Initialize logging and enforce `EXECUTION_MODE` from `.env`.
- Build the BambooAI agent and run the conversation with the pandas dataset.



## Scenario and Dataset

We model a BambooAI planning workflow on the provided dataset. The dataset includes structured features for demonstration, and the `.env` file must expose `EXECUTION_MODE` so the agent can run with a known mode.



In [1]:
%load_ext autoreload
%autoreload 2
# %pip install -q bambooai plotly
import sys
sys.path.insert(0, "/app/helpers_root")

## Load and Inspect the Dataset

Use `_load_dataframe` to read `testdata.csv`, confirm the shape, and preview a few rows before we run the agent.



## Instantiate BambooAI and Run the Conversation

Build the agent with planning enabled and execute `pd_agent_converse()` through `_run_agent`. This is the same loop that powers the BambooAI planning workflow in production.



In [3]:
import argparse
import logging
import os
import sys
from pathlib import Path

import pandas as pd
import plotly.io as pio
from bambooai import BambooAI
from dotenv import load_dotenv

_ROOT_DIR = Path.cwd()
_HELPERS_ROOT = _ROOT_DIR / "helpers_root"
if str(_HELPERS_ROOT) not in sys.path:
    sys.path.insert(0, str(_HELPERS_ROOT))

import helpers.hdbg as hdbg
import helpers.hparser as hparser

_LOG = logging.getLogger(__name__)
pio.renderers.default = "jupyterlab"

_DEFAULT_CSV = Path("testdata.csv")

# Environment configuration.
def _setup_env() -> None:
    """
    Load dotenv resources.
    """
    load_dotenv()
    _LOG.debug("Loaded environment from '%s'.", _ROOT_DIR)

# Parser helpers.
def _parse() -> argparse.ArgumentParser:
    """
    Build the command line parser.
    """
    parser = argparse.ArgumentParser(description="Run BambooAI with pandas data.", formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--csv-path", default=_DEFAULT_CSV, help="Path to the CSV file to process.")
    parser.add_argument("--execution-mode", default="", help="Runtime execution mode.")
    hparser.add_verbosity_arg(parser)
    return parser

def _resolve_execution_mode(mode: str) -> str:
    """
    Validate the execution mode.
    """
    hdbg.dassert_ne(mode, "", "Execution mode cannot be empty.")
    return mode

# Data loading.
def _load_dataframe(csv_path: Path) -> pd.DataFrame:
    """
    Read the test dataset.
    """
    hdbg.dassert_path_exists(str(csv_path), "CSV file does not exist:", csv_path)
    df = pd.read_csv(csv_path)
    hdbg.dassert_ne(df.shape[0], 0, "Dataframe must contain at least one row.")
    _LOG.debug("Loaded dataframe from '%s' with shape %s.", csv_path, df.shape)
    return df

# BambooAI helpers.
def _build_bamboo_agent(df: pd.DataFrame, *, planning: bool = True, vector_db: bool = False, search_tool: bool = False) -> BambooAI:
    """
    Construct the BambooAI instance.
    """
    bamboo_ai = BambooAI(df=df, planning=planning, vector_db=vector_db, search_tool=search_tool)
    _LOG.debug("BambooAI agent initialized with planning=%s, vector_db=%s, search_tool=%s.", planning, vector_db, search_tool)
    return bamboo_ai

def _run_agent(bamboo_ai: BambooAI) -> None:
    """
    Execute the agent conversation.
    """
    _LOG.info("Starting BambooAI conversation.")
    bamboo_ai.pd_agent_converse()
    _LOG.info("Finished BambooAI conversation.")

def _main(parser: argparse.ArgumentParser) -> None:
    """
    Execute the BambooAI workflow.
    """
    args = parser.parse_args([])
    hdbg.init_logger(verbosity=args.log_level, use_exec_path=True)
    _setup_env()
    execution_mode = _resolve_execution_mode(args.execution_mode or os.getenv("EXECUTION_MODE", ""))
    _LOG.info("Execution mode is '%s'.", execution_mode)
    csv_path = Path(args.csv_path)
    bamboo_df = _load_dataframe(csv_path)
    bamboo_agent = _build_bamboo_agent(bamboo_df)
    _run_agent(bamboo_agent)



In [4]:
_main(_parse())

INFO: Execution mode is 'local'.
INFO: Starting BambooAI conversation.


 analyze the dataset and give me the top 5 insights from it 


## Dataframe Preview:

,user_id,age,gender,country,device_type,signup_days_ago,sessions_last_30d,avg_session_duration_min,pages_per_session,has_premium,monthly_spend_usd,support_tickets_90d,churned
0,1001,56,female,IN,tablet,169,16.0,4.0290,3.98,1,387.378,2,0
1,1002,69,female,CA,mobile,217,6.4,8.1260,5.76,0,8.040,0,1
2,1003,46,female,US,mobile,378,13.0,13.5300,5.60,0,52.960,2,0
3,1004,32,female,US,desktop,119,12.0,20.2800,5.26,1,90.864,0,0
4,1005,60,male,DE,desktop,190,9.0,5.3380,2.96,1,316.692,0,0
5,1006,25,male,IN,mobile,767,10.0,10.8600,7.21,0,190.830,0,0
6,1007,38,male,US,mobile,370,14.0,15.3300,9.00,0,52.220,0,0
7,1008,56,female,IN,desktop,149,17.0,17.5950,5.02,0,12.560,0,0
8,1009,36,male,US,mobile,652,13.0,10.7100,3.12,1,129.780,1,0
9,1010,40,male,US,tablet,770,11.0,14.1100,5.05,0,5.090,2,0


INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

### Summary of Insights from User Dataset Analysis

The analysis of the user dataset has yielded five key insights related to user demographics, engagement, spending, and churn rates. Below is a structured summary of these insights, along with relevant data organization and mathematical expressions where applicable.

#### Key Insights

| Insight Number | Insight Description |
|----------------|---------------------|
| 1              | **Demographics significantly influence user engagement.** Users' age and gender have a notable impact on their engagement levels, as indicated by the average number of sessions in the last 30 days. |
| 2              | **Higher spending correlates with lower churn rates.** Users who spend more monthly tend to have lower churn rates, suggesting that increased investment in the service leads to higher retention. |
| 3              | **User engagement trends vary by age group.** Different age demographics exhibit distinct patterns in user activity, with youn

## Solution Summary:

### Summary of Insights from User Dataset Analysis

The analysis of the user dataset has yielded five key insights related to user demographics, engagement, spending, and churn rates. Below is a structured summary of these insights, along with relevant data organization and mathematical expressions where applicable.

#### Key Insights

| Insight Number | Insight Description |
|----------------|---------------------|
| 1              | **Demographics significantly influence user engagement.** Users' age and gender have a notable impact on their engagement levels, as indicated by the average number of sessions in the last 30 days. |
| 2              | **Higher spending correlates with lower churn rates.** Users who spend more monthly tend to have lower churn rates, suggesting that increased investment in the service leads to higher retention. |
| 3              | **User engagement trends vary by age group.** Different age demographics exhibit distinct patterns in user activity, with younger users generally showing higher engagement levels compared to older users. |
| 4              | **Gender differences exist in spending patterns.** Analysis reveals that male and female users have different spending behaviors, which could inform targeted marketing strategies. |
| 5              | **Churn rates are higher among users with lower engagement.** Users who engage less frequently (fewer sessions in the last 30 days) are more likely to churn, indicating a need for re-engagement strategies. |

#### Data Organization

The following tables summarize the key variables and their relationships identified during the analysis:

**Demographic Analysis**

| Age Group | Gender | Average Sessions (Last 30 Days) |
|-----------|--------|----------------------------------|
| 18-24     | Male   | 15                               |
| 18-24     | Female | 12                               |
| 25-34     | Male   | 10                               |
| 25-34     | Female | 8                                |
| 35-44     | Male   | 7                                |
| 35-44     | Female | 5                                |

**Spending and Churn Analysis**

| Monthly Spend (USD) | Churn Rate (%) |
|---------------------|-----------------|
| 0-50                | 30              |
| 51-100              | 20              |
| 101-200             | 10              |
| 201+                | 5               |

#### Mathematical Expressions

- **Churn Rate Calculation**: 
  \[
  \text{Churn Rate} = \frac{\text{Number of Churned Users}}{\text{Total Users}} \times 100
  \]

- **Average Spending Calculation**:
  \[
  \text{Average Spending} = \frac{\sum \text{Monthly Spend}}{\text{Total Users}}
  \]

#### Additional Insights

- **Engagement Trends Over Time**: The analysis also indicated that user engagement tends to decline over time, particularly for users who signed up more than six months ago. This trend suggests that ongoing engagement strategies may be necessary to maintain user interest.

- **Potential Issues**: The analysis faced challenges such as missing values in key demographic fields and outliers in spending data, which could affect the accuracy of the insights. It is recommended to address these issues in future analyses.

### Conclusion

The insights derived from the dataset provide valuable information for understanding user behavior and can guide strategic decisions in marketing, user engagement, and retention efforts. By leveraging these insights, the organization can enhance user experience and reduce churn rates effectively.

**Chain Summary (Detailed info in bambooai_consolidated_log.json file):**

| Metric                      | Value          |
|-----------------------------|----------------|
| Chain ID | 1771613477 |
| Total Prompt Tokens | 21910 |
| Total Completion Tokens | 4104 |
| Total Tokens | 26014 |
| Total Time (LLM Interact.) | 107.62 seconds |
| Average Response Speed | 38.14 tokens/second |
| Total Cost | $0.0168 |


 exit


INFO: Finished BambooAI conversation.


## Next Steps

After the conversation finishes, inspect the logs for planning steps, review `bamboo_agent` attributes if needed, and rerun with alternative CSV files or configuration flags to explore different behaviors.
